# Addestramento SFCN - Adattamento Dataset IXI (T2)
Questo notebook contiene la pipeline di addestramento modificata per gestire la struttura del nuovo dataset IXI e l'estrazione dell'età dal file CSV (sottraendo la data di nascita alla data di riferimento del 23 Febbraio 2015).

In [ ]:
!rm -rf SFCN
!git clone https://github.com/PietroSchgor/SFCN.git

import sys
sys.path.append('./SFCN')

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import nibabel as nib
import torch
import torch.nn as nn
from datetime import datetime
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import StratifiedKFold, train_test_split

from dp_model.model_files.sfcn import SFCN
from train import train_model
from dp_model import dp_utils as dpu

## 1. Configurazione Parametri
**IMPORTANTE**: Nel dataset IXI l'età massima supera abbondantemente gli 80 anni (fino a 91). Di conseguenza, dobbiamo espandere i bin della rete neurale da `70` a `100`!

In [ ]:
KAGGLE_DATA_DIR = "/kaggle/input/datasets/collab4444/dataset-2-t1-t2/Prep_IXI_T2/Prep_IXI_T2"

# Se esegui il notebook su Kaggle e hai caricato il CSV lì, aggiorna questo percorso!
# Altrimenti lascialo F:\test\ixi_info.csv se lo fai girare in locale.
CSV_PATH = "F:\\test\\ixi_info.csv"

OUTPUT_DIM = 100  # Età massima supportata: 100 anni

EPOCHS = 130
BATCH_SIZE = 4
LR = 1e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 15

## 2. Definizione del Dataset IXI

In [ ]:
class IXIBrainAgeDataset(Dataset):
    def __init__(self, data_dir, csv_path, is_train=False):
        self.data_dir = data_dir
        self.is_train = is_train
        self.samples = []
        
        self.bin_range = [0, OUTPUT_DIM]
        self.bin_step = 1
        self.sigma = 1.0
        
        # Caricamento CSV e Data di Riferimento
        df = pd.read_csv(csv_path)
        reference_date = datetime(2015, 2, 23)
        
        # Sottocartelle note nel dataset
        subfolders = ['Prep_Guys_T2', 'Prep_HH_T2', 'Prep_IOP_T2']
        
        for sf in subfolders:
            folder_path = os.path.join(data_dir, sf)
            if not os.path.exists(folder_path):
                continue
                
            for file in os.listdir(folder_path):
                if file.startswith("registered_image_") and file.endswith(".nii"):
                    nii_path = os.path.join(folder_path, file)
                    
                    # Estrai ID dal nome file (es. registered_image_100.nii -> 100)
                    ixi_id_str = file.replace("registered_image_", "").replace(".nii", "")
                    try:
                        ixi_id = int(ixi_id_str)
                    except ValueError:
                        continue
                        
                    # Match con il CSV
                    row = df[df['IXI_ID'] == ixi_id]
                    if len(row) == 0:
                        continue
                        
                    dob_str = row.iloc[0]['DOB']
                    if pd.isna(dob_str):
                        continue
                        
                    try:
                        # Rimuove eventuali orari se presenti nel CSV (es. 1965-03-08 00:00:00)
                        dob_str = str(dob_str).split(" ")[0]
                        dob = datetime.strptime(dob_str, "%Y-%m-%d")
                        
                        # Calcolo età esatta al 23 Feb 2015
                        true_age = (reference_date - dob).days / 365.25
                        
                        # Generazione Soft-Label
                        y, _ = dpu.num2vect(true_age, self.bin_range, self.bin_step, self.sigma)
                        
                        self.samples.append({
                            "nii_path": nii_path,
                            "label_vect": y,
                            "true_age": true_age
                        })
                    except Exception as e:
                        continue
                        
        print(f"[{'TRAIN' if is_train else 'TEST/VAL'}] Caricati {len(self.samples)} pazienti IXI validi.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        img = nib.load(sample['nii_path'])
        data = img.get_fdata(dtype=np.float32)
        
        # Normalizzazione
        mean_val = np.mean(data)
        if mean_val > 0:
            data = data / mean_val
            
        # Cropping/Padding (Dimensione attesa da SFCN: 160x192x160)
        in_sp = data.shape
        out_sp = (160, 192, 160)
        
        if self.is_train:
            dx = np.random.randint(-3, 4)
            dy = np.random.randint(-3, 4)
            dz = np.random.randint(-3, 4)
        else:
            dx, dy, dz = 0, 0, 0
            
        x_c = int((in_sp[0] - out_sp[0]) / 2) + dx
        y_c = int((in_sp[1] - out_sp[1]) / 2) + dy
        z_c = int((in_sp[2] - out_sp[2]) / 2) + dz
        
        data = data[x_c:x_c+out_sp[0], y_c:y_c+out_sp[1], z_c:z_c+out_sp[2]]
        
        # Data Augmentation
        if self.is_train and np.random.rand() > 0.5:
            data = np.flip(data, axis=0).copy()
            
        data = np.expand_dims(data, axis=0)
        
        tensor_data = torch.from_numpy(data)
        label_vect = torch.tensor(sample['label_vect'], dtype=torch.float32)
        
        return tensor_data, label_vect, sample['true_age']

## 3. Preparazione Dati e Split K-Fold

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device in uso: {device}")

full_train_dataset = IXIBrainAgeDataset(KAGGLE_DATA_DIR, CSV_PATH, is_train=True)
full_val_dataset = IXIBrainAgeDataset(KAGGLE_DATA_DIR, CSV_PATH, is_train=False)

dataset_size = len(full_train_dataset)

if dataset_size > 0:
    all_ages = [sample['true_age'] for sample in full_train_dataset.samples]
    all_indices = np.arange(dataset_size)
    
    # Blocchiamo il 20% come Test Set Finale
    train_val_idx, test_idx = train_test_split(
        all_indices, 
        test_size=0.20, 
        random_state=42
    )
    
    # Il restante 80% verrà diviso nei 5 Fold
    train_val_ages = np.array(all_ages)[train_val_idx]
    
    print(f"Pazienti per K-Fold (80%): {len(train_val_idx)}")
    print(f"Pazienti per Test Finale (20%): {len(test_idx)}")
else:
    print("ERRORE: Impossibile trovare i file. Controlla KAGGLE_DATA_DIR e CSV_PATH.")

## 4. Addestramento K-Fold Ensemble

In [ ]:
NUM_FOLDS = 5
fold_results = []
saved_models_paths = []

if dataset_size > 0:
    skf = StratifiedKFold(n_splits=NUM_FOLDS, shuffle=True, random_state=42)
    
    for fold, (train_split_idx, val_split_idx) in enumerate(skf.split(train_val_idx, [int(a/5) for a in train_val_ages])):
        print(f"\n{'='*50}")
        print(f" INIZIO ADDESTRAMENTO FOLD {fold+1}/{NUM_FOLDS} (IXI T2)")
        print(f"{'='*50}")
        
        fold_train_idx = train_val_idx[train_split_idx]
        fold_val_idx = train_val_idx[val_split_idx]
        
        train_dataset = torch.utils.data.Subset(full_train_dataset, fold_train_idx)
        val_dataset = torch.utils.data.Subset(full_val_dataset, fold_val_idx)
        
        # Ribilanciamento con WeightedRandomSampler
        fold_ages = [all_ages[i] for i in fold_train_idx]
        fold_age_classes = [int(a/5) for a in fold_ages]
        class_counts = np.bincount(fold_age_classes)
        
        weights = []
        for c in fold_age_classes:
            w = 1.0 / class_counts[c] if class_counts[c] > 0 else 0
            weights.append(w)
            
        sampler = WeightedRandomSampler(
            weights=weights, 
            num_samples=len(weights), 
            replacement=True
        )
        
        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2)
        val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=2)
        
        # Inizializziamo il modello (NOTA: output_dim=100)
        model = SFCN(output_dim=OUTPUT_DIM)
        if torch.cuda.device_count() > 1:
            model = nn.DataParallel(model)
        model = model.to(device)
        
        optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        
        steps_per_epoch = len(train_loader)
        
        trained_model, t_losses, v_losses, v_maes = train_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            optimizer=optimizer,
            device=device,
            epochs=EPOCHS,
            step_size=steps_per_epoch * 30,
            gamma=0.3,
            patience=PATIENCE,
            fold_idx=fold+1
        )
        
        best_fold_mae = min(v_maes)
        fold_results.append(best_fold_mae)
        
        fold_save_path = f"/kaggle/working/sfcn_IXI_T2_fold_{fold+1}.pth"
        if isinstance(trained_model, nn.DataParallel):
            torch.save(trained_model.module.state_dict(), fold_save_path)
        else:
            torch.save(trained_model.state_dict(), fold_save_path)
            
        saved_models_paths.append(fold_save_path)

    print("\n==============================================")
    print("   RIASSUNTO ADDESTRAMENTO 5-FOLD (IXI T2)")
    print("==============================================")
    for i, mae in enumerate(fold_results):
        print(f"Fold {i+1} Miglior Val MAE: {mae:.2f} anni")
        
    best_fold_idx = np.argmin(fold_results)
    print(f"\n>>> Il Miglior Fold singolo è stato il Fold {best_fold_idx+1} con Val MAE: {fold_results[best_fold_idx]:.2f}")

## 5. Valutazione sul Test Set Finale (100 Bin)

In [ ]:
if dataset_size > 0:
    test_dataset = torch.utils.data.Subset(full_val_dataset, test_idx)
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)
    
    # Ricarichiamo in memoria tutti i 5 modelli
    ensemble_models = []
    for path in saved_models_paths:
        m = SFCN(output_dim=OUTPUT_DIM)
        m.load_state_dict(torch.load(path, map_location=device))
        m.to(device)
        m.eval()
        ensemble_models.append(m)

    all_test_true_ages = []
    all_test_preds_best_fold = []
    all_test_preds_ensemble = []

    bin_centers = np.arange(0, OUTPUT_DIM, 1)

    with torch.no_grad():
        for inputs, _, true_age in test_loader:
            inputs = inputs.to(device)
            true_age_val = true_age.item()
            all_test_true_ages.append(true_age_val)
            
            probabilities_from_all_folds = []
            
            for m in ensemble_models:
                out = m(inputs)[0].view(1, -1)
                prob = torch.exp(out).cpu().numpy()
                probabilities_from_all_folds.append(prob)
                
            # Predizione dell'Ensemble
            avg_prob = np.mean(probabilities_from_all_folds, axis=0)
            ensemble_pred_age = (avg_prob @ bin_centers)[0]
            all_test_preds_ensemble.append(ensemble_pred_age)
            
            # Predizione del Miglior Singolo Fold
            best_fold_prob = probabilities_from_all_folds[best_fold_idx]
            best_fold_pred_age = (best_fold_prob @ bin_centers)[0]
            all_test_preds_best_fold.append(best_fold_pred_age)
            
    mae_best_fold = np.mean(np.abs(np.array(all_test_preds_best_fold) - np.array(all_test_true_ages)))
    mae_ensemble = np.mean(np.abs(np.array(all_test_preds_ensemble) - np.array(all_test_true_ages)))
    
    print("\n==============================================")
    print("     RISULTATI FINALI SUL TEST SET IXI")
    print("==============================================")
    print(f"MAE Miglior Singolo Modello (Fold {best_fold_idx+1}) : {mae_best_fold:.3f} anni")
    print(f"MAE K-Fold Ensemble (Media di 5)     : {mae_ensemble:.3f} anni")
    print("==============================================\n")
